# Sampling flux vacua

**What's in this notebook?** This notebook demonstrates how to sample flux vacua using a *flux-first* approach: given a set of input fluxes, it searches for all moduli configurations giving rise to a SUSY minimum. This complements the moduli-first approach in notebook [8](8_ISD_sampling_wrapper.ipynb).

**In this notebook, you will learn:**
- How to define input fluxes and use `vmapping_func` to build doubly-vmapped optimisers
- The two-step algorithm: ISD initial guesses via `find_solution_init_vmap`, followed by Newton refinement via `find_solution_steps_vmap`
- How to use `model.sample_SUSY_vacua_from_fluxes` for large-scale flux-first searches
- Constrained sampling: restricting to specific regions of moduli space for a given flux choice

**Use case:** This approach is particularly useful when you want to systematically enumerate all vacua associated with a specific set of flux quanta, as done in [2501.03984](https://arxiv.org/abs/2501.03984).

**Comparison with notebook 8:**

| | [Notebook 8](8_ISD_sampling_wrapper.ipynb) | This notebook |
|---|---|---|
| Starting point | Random moduli → ISD → fluxes | Fixed input fluxes → ISD → moduli |
| Use case | Broad flux vacuum distribution | Systematic search for given fluxes |
| Main function | `sample_SUSY_flux_vacua` | `sample_SUSY_vacua_from_fluxes` |
| Double vmap | No | Yes (over fluxes × starting points) |

**Related notebooks:** [04_sampling_module](../01_basics/04_sampling_module.ipynb) (ISD sampling concepts), [8_ISD_sampling_wrapper](8_ISD_sampling_wrapper.ipynb) (moduli-first approach), [10b_stochastic_flux_search](../03_flux_bounding/10b_stochastic_flux_search.ipynb) (stochastic flux enumeration).

(*Created:* Andreas Schachner, June 25, 2024)

## Imports

In [ ]:
# General imports
import warnings, sys, time
import numpy as np
from tqdm.auto import tqdm

# JAX imports
from jax import jit, vmap
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

# Plotting
import seaborn as sn
import matplotlib.pyplot as plt
cmap = sn.color_palette("viridis", as_cmap=True)

# JAXVacua
import jaxvacua as jvc
from jaxvacua.util import vmapping_func

warnings.filterwarnings('ignore')

## Plotting function

In [ ]:
sys.path.append("../")
from plot_utils import make_overview_plots

## Sampling from input fluxes

Load model from files

In [ ]:
#if False:
if True:
    h12=2
    model = jvc.FluxVacuaFinder(h12 = h12, Q=276, model_ID = 1, maximum_degree = 2,limit="LCS",model_type="KS",map_to_fd=True)

Alternatively, use CYTools

In [ ]:
from cytools import Polytope, Cone, fetch_polytopes, read_polytopes

if False:
    p = fetch_polytopes(h11=2,h12=272,limit=5,lattice="N",as_list=True)[0]
    cy = p.triangulate().get_cy()
    mcap = cy.mori_cone_cap(in_basis=True)
    Kcup = mcap.dual_cone()
    basis_trafo = Kcup.extremal_rays()

    model = jvc.FluxEFT(h12=cy.h11(), Q=cy.h11()+cy.h12()+2, model_type="KS", maximum_degree=2, 
                                      use_cytools=True, mirror_cy = cy, basis_transformation=basis_trafo)


In contrast to the sampling method described in the notebook [ISD_sampling_wrapper](./ISD_sampling_wrapper.ipynb),
this approach takes as input a set of fluxes and tries to find points in moduli space giving rise to a suitable
flux vacuum via the ISD sampling method.

This version of the code was used in [2501.03984](https://arxiv.org/abs/2501.03984) to systematically construct all flux vacua
in a certain region of moduli space. Note, however, that we are currently not providing an implementation for the bounds
required to find the choices of input fluxes.



The idea of the implementation is pretty simple and will be illustrated with a simple toy example below.
In short, the implementation requires the following steps:
* we define the input fluxes as **half** of the total number of fluxes,
* `vmap` fluxes across many starting points (that is, for one flux, there will be many starting points),
* determine the remaining fluxes via ISD sampling, and
* solve $F$-term conditions for the corresponding choices of fluxes and initial guesses.

#### The algorithm

Let us define some parameters for the optimisation

In [ ]:
mode = "Fflux"
kwargs = {"mode":mode,"Q":model.D3_tadpole,"constraints":None,"remove_NANs":True}

The objective function `model.DW` needs to be vmapped twice

In [ ]:
DW_vv = jax.vmap(jax.vmap(model.DW))

We have to define two separate functions:

In [ ]:
find_solution_init = vmapping_func(model.linearised_shifts,in_axes=(0,0,None),return_flag=False,**kwargs)
find_solution_init_vmap = vmapping_func(find_solution_init,in_axes=(None,None,0))

The former `find_solution_init` maps different starting guesses across a single flux.
The second optimiser `find_solution_init_vmap` is used to apply this map across **all fluxes**.

Our input of starting guesses and fluxes is defined as

In [ ]:
moduli0 = np.array([[ 0.22924045+3.00550378j, -0.46016314+2.28651085j],
        [-0.04468382+0.63834065j,  0.25839392+2.36493542j]])
tau0 = np.array([0.18254317+6.81361114j, 0.49828078+8.33716844j])
fluxes0 = np.array([[ 1,  5,  1,  2,  4, -1],
                    [ 5, -3,  0, -4, -4, -3]])

The first step is then to apply  

In [ ]:
moduli,tau,fluxes = find_solution_init_vmap(moduli0,tau0,fluxes0)
moduli,tau,fluxes

Next, we need an optimiser to evolve the starting guesses to a solution of $D_IW=0$.
This is achieved by defining

In [ ]:
find_solution_steps = vmapping_func(model.linearised_shifts,in_axes=(0,0,0),return_flag=True,**kwargs)
find_solution_steps_vmap = vmapping_func(find_solution_steps,in_axes=(0,0,0))

We can then apply `find_solution_steps_vmap` to find the next step towards a minimum of the potential

In [ ]:
moduli1,tau1,fluxes1,checks = find_solution_steps_vmap(moduli,tau,fluxes)

To show that the new values are closer to a solution to $D_IW=0$, let us evalute $D_IW$ before and after this optimisation step:

In [ ]:
DW0 = DW_vv(moduli,jnp.conj(moduli),tau,jnp.conj(tau),fluxes)
DW1 = DW_vv(moduli1,jnp.conj(moduli1),tau1,jnp.conj(tau1),fluxes1)
np.sum(np.abs(DW1),axis=2)/np.sum(np.abs(DW0),axis=2)

We see that the ratio of the new over the old $F$-terms is smaller than one, so we are actually moving towards a minimum.
Let us apply several 

In [ ]:
for i in range(25):
    moduli1,tau1,fluxes1,checks = find_solution_steps_vmap(moduli1,tau1,fluxes1)
    DW = DW_vv(moduli1,jnp.conj(moduli1),tau1,jnp.conj(tau1),fluxes1)
    res = np.sum(np.abs(DW),axis=2)
    print(f"step: {i}       |DW| for flux1: {res[0]}              |DW| for flux2: {res[1]}                   ")

The residuals for two choices of fluxes and starting guesses led us to a minimum, while the other ones did not yield anything useful.

#### Large scale search for minima

**WORK IN PROGRESS**

In [ ]:
mode = "Fflux"
kwargs = {"mode":mode,"Q":model.D3_tadpole,"constraints":None,"remove_NANs":True}

find_solution_init = vmapping_func(model.linearised_shifts,in_axes=(0,0,None),return_flag=False,**kwargs)
find_solution_init_vmap = vmapping_func(find_solution_init,in_axes=(None,None,0))
find_solution_steps = vmapping_func(model.linearised_shifts,in_axes=(0,0,0),return_flag=True,**kwargs)
find_solution_steps_vmap = vmapping_func(find_solution_steps,in_axes=(0,0,0))

In [ ]:
seed = 42
rns_key = jvc.PRNGSequence(seed)

In [ ]:
DW_vv = jax.vmap(jax.vmap(model.DW))

In [ ]:
sampler = jvc.data_sampler(model,flux_bounds=[-5,5],moduli_bounds=[2,3],dilaton_bounds=[4,10])

In [ ]:
fluxes0 = sampler.get_fluxes(100)

fluxes0 = fluxes0[:,:model.n_fluxes]

We run the following line to find flux vacua: (**this can take a few minutes**)

In [ ]:
moduli,tau,fluxes,residual = model.sample_SUSY_vacua_from_fluxes(
                                    fluxes_init=fluxes0,
                                    N=2*10**3,
                                    sampler = sampler,
                                    rns_key=rns_key,
                                    max_iters = 10,
                                    objective_fct = DW_vv, 
                                    optimiser_init=find_solution_init_vmap,optimiser_steps=find_solution_steps_vmap,
                                    vmap_dim_pts = 10,
                                    mode=mode,print_progress=True)

Let us check that `DW` is indeed small

In [ ]:
DW = jax.vmap(model.DW)(moduli,jnp.conj(moduli),tau,jnp.conj(tau),fluxes)
np.sum(np.abs(DW),axis=1),np.where(np.sum(np.abs(DW),axis=1)>1e-10)

All solution satisfy $\sum |DW|<10^{-10}$!


We can check the number of individual flux components as follows:

In [ ]:
f1,f2,h1,h2 = vmap(model._split_fluxes)(fluxes)
np.unique(f1,axis=0).shape,np.unique(f2,axis=0).shape,np.unique(h1,axis=0).shape,np.unique(h2,axis=0).shape

Note that our choice of input fluxes `fluxes0` in `mode = "Fflux"` corresponds to the RR-fluxes `f1` and `f2`.
Since we started with an input of 100 choices `(f1,f2)`, the number of flux choices for `f1` and `f2` is obviously less than that.

Let us plot the distribution of flux vacua:

In [ ]:
make_overview_plots(model,moduli,tau,fluxes,moduli_range=[[-0.7,0.7],[1,4]],tau_range=[[-0.5,0.5],[3,12]],W0_range=[-10,10])

### Improved constrained sampling -- regions of moduli space with given fluxes

Sample from a given set of fluxes across many points in moduli space

**WORK IN PROGRESS**

In [ ]:
@jit
def constraints_model(moduli,tau,flux):
    
    return ((jnp.all(jnp.imag(moduli)<3.,axis=0))&(jnp.all(jnp.imag(moduli)>2.,axis=0))&(jnp.imag(tau)>4))

mode = "Fflux"
kwargs = {"mode":mode,"Q":model.D3_tadpole,"constraints":constraints_model,"remove_NANs":True}

find_solution_init = vmapping_func(model.linearised_shifts,in_axes=(0,0,None),return_flag=False,**kwargs)
find_solution_init_vmap = vmapping_func(find_solution_init,in_axes=(None,None,0))
find_solution_steps = vmapping_func(model.linearised_shifts,in_axes=(0,0,0),return_flag=True,**kwargs)
find_solution_steps_vmap = vmapping_func(find_solution_steps,in_axes=(0,0,0))

In [ ]:
sampler = jvc.data_sampler(model,flux_bounds=[-5,5],moduli_bounds=[2,3],dilaton_bounds=[4,10])

In [ ]:
seed = 42
rns_key = jvc.PRNGSequence(seed)

In [ ]:
vmap_dim = 10
moduli_sampling_mode = "cone"
moduli,tau = sampler.initial_guesses(vmap_dim,rns_key=rns_key,moduli_sampling_mode=moduli_sampling_mode,include_fluxes=False)

In [ ]:
DW_vv = jax.vmap(jax.vmap(model.DW))

In [ ]:
moduli,tau,fluxes,residuals = model.sample_SUSY_vacua_from_fluxes(fluxes_init=None,N=100,sampler = sampler,rns_key=rns_key,max_iters = 10,moduli_sampling_mode = "cone",max_tadpole = 276,
                           objective_fct = DW_vv, optimiser_init=find_solution_init_vmap,optimiser_steps=find_solution_steps_vmap,
                            vmap_dim_flux = 100,vmap_dim_pts = 100,mode=mode,print_progress=True)

In [ ]:
make_overview_plots(model,moduli,tau,fluxes,moduli_range=[[-0.5,0.5],[1,4]],tau_range=[[-0.5,0.5],[3,12]],W0_range=[-10,10])

## Summary

This notebook demonstrated the flux-first sampling approach via `model.sample_SUSY_vacua_from_fluxes`. Key takeaways:

- The two-level vmap (`find_solution_init_vmap`, `find_solution_steps_vmap`) efficiently handles many flux–starting-point combinations simultaneously.
- For a given set of input fluxes, ISD sampling produces moduli initial guesses; Newton refinement then converges to the actual SUSY minimum.
- Not all (flux, starting-point) combinations converge — this is expected; diverging trajectories are filtered out automatically.
- The constrained variant allows restricting moduli to specific regions while still searching exhaustively over the input flux set.

**Next steps:**
- [8_ISD_sampling_wrapper](8_ISD_sampling_wrapper.ipynb) — moduli-first sampling approach
- [10_flux_bounding](../03_flux_bounding/10_flux_bounding.ipynb) — analytically bounded flux lattice enumeration
- [10b_stochastic_flux_search](../03_flux_bounding/10b_stochastic_flux_search.ipynb) — stochastic search over the bounded flux lattice